In [0]:
%run ./00_Config_and_Logging

In [0]:
# ==========================================
# 02_Bronze_to_Silver: Silver data model (explicit schema)
# Primary key: (city, reading_time)
# ==========================================
import os
from datetime import datetime, date, timedelta
from functools import reduce
from operator import or_
from delta.tables import DeltaTable
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

silver_quarantine_path = f"{os.path.dirname(silver_delta_path)}/silver_quarantine"

silver_schema = StructType([
    StructField("city", StringType(), False),             # PK
    StructField("reading_time", TimestampType(), False),  # PK
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("elevation", DoubleType(), True),
    StructField("timezone", StringType(), True),
    StructField("carbon_monoxide_ugm3", DoubleType(), True),
    StructField("carbon_dioxide_ppm", DoubleType(), True),
    StructField("pm2_5_ugm3", DoubleType(), True),
    StructField("pm10_ugm3", DoubleType(), True),
    StructField("nitrogen_dioxide_ugm3", DoubleType(), True),
    StructField("source_file", StringType(), True),            # lineage: which raw file
    StructField("bronze_load_timestamp", TimestampType(), True),
    StructField("load_timestamp", TimestampType(), False),     # when this row was written to Silver
])

# Rejected rows: same columns (all nullable) + why they were rejected
silver_quarantine_schema = StructType(
    [StructField(f.name, f.dataType, True) for f in silver_schema.fields if f.name != "load_timestamp"]
    + [StructField("quarantine_reason", StringType(), True),
       StructField("load_timestamp", TimestampType(), True)])

def ensure_table(path, schema):
    """Create the empty Delta table if missing (MERGE needs a target); refuse an outdated layout."""
    try:
        cols = spark.read.format("delta").load(path).columns
        if cols != schema.fieldNames():
            raise Exception(f"OLD_LAYOUT: table at {path} has outdated columns. Run once, then re-run:\n"
                            f'    dbutils.fs.rm("{path}", recurse=True)')
        print("Table ready:", path)
    except Exception as e:
        if "OLD_LAYOUT" in str(e):
            raise
        spark.createDataFrame([], schema).write.format("delta").save(path)
        print("Table created:", path)

ensure_table(silver_delta_path, silver_schema)
ensure_table(silver_quarantine_path, silver_quarantine_schema)

# Watermark: newest Bronze load_timestamp that Silver has already processed.
# An incremental run only takes Bronze rows NEWER than this, so nothing is missed or re-read.
silver_watermark_path = f"{os.path.dirname(silver_delta_path)}/silver_watermark"
silver_watermark_schema = StructType([
    StructField("processed_up_to", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)])
ensure_table(silver_watermark_path, silver_watermark_schema)

def get_watermark():
    return spark.read.format("delta").load(silver_watermark_path).agg(F.max("processed_up_to")).first()[0]

def set_watermark(ts):
    (spark.createDataFrame([(ts, datetime.now())], silver_watermark_schema)
          .write.format("delta").mode("append").save(silver_watermark_path))

In [0]:
# ==========================================
# Transformation: window -> explode -> cast -> validate
# ==========================================
POLLUTANTS = ["carbon_monoxide_ugm3", "carbon_dioxide_ppm", "pm2_5_ugm3", "pm10_ugm3", "nitrogen_dioxide_ugm3"]
VALUE_COLS = ["latitude", "longitude", "elevation", "timezone"] + POLLUTANTS

def clean_and_explode(bronze_df, start_date=None, end_date=None):
    """One row per city-hour (optionally limited to a READING-date window), typed and flagged.
    dq_reason is NULL for valid rows and a reason code for rejected rows."""
    zipped = F.arrays_zip(
        col("hourly.time").alias("time"),
        col("hourly.carbon_monoxide").alias("carbon_monoxide"),
        col("hourly.carbon_dioxide").alias("carbon_dioxide"),
        col("hourly.pm2_5").alias("pm2_5"),
        col("hourly.pm10").alias("pm10"),
        col("hourly.nitrogen_dioxide").alias("nitrogen_dioxide"),
    )
    if start_date and end_date:   # backfill: keep only hours inside the window BEFORE exploding
        lo, hi = f"{start_date}T00:00", f"{end_date}T23:59"
        in_window = F.filter(zipped, lambda r: (r["time"] >= lo) & (r["time"] <= hi))
    else:                         # incremental: every hour of the new Bronze rows
        in_window = zipped

    exploded = (bronze_df
        .select("city", "latitude", "longitude", "elevation", "timezone", "source_file",
                col("load_timestamp").alias("bronze_load_timestamp"),
                F.explode(in_window).alias("reading"))
        .select("city", "latitude", "longitude", "elevation", "timezone", "source_file",
                "bronze_load_timestamp",
                F.to_timestamp(col("reading.time")).alias("reading_time"),
                col("reading.carbon_monoxide").cast("double").alias("carbon_monoxide_ugm3"),
                col("reading.carbon_dioxide").cast("double").alias("carbon_dioxide_ppm"),
                col("reading.pm2_5").cast("double").alias("pm2_5_ugm3"),
                col("reading.pm10").cast("double").alias("pm10_ugm3"),
                col("reading.nitrogen_dioxide").cast("double").alias("nitrogen_dioxide_ugm3")))

    sentinel = reduce(or_, [col(c) == -999 for c in POLLUTANTS])
    negative = reduce(or_, [col(c) < 0 for c in POLLUTANTS]) | (col("carbon_dioxide_ppm") == 0)
    reason = (F.when(col("reading_time").isNull(), "INVALID_TIMESTAMP")
               .when(col("city").isNull(), "NULL_CITY")
               .when(sentinel, "SENTINEL_VALUE_-999")
               .when(negative, "NEGATIVE_OR_ZERO_VALUE"))
    return exploded.withColumn("dq_reason", reason)


def dedupe_latest(df):
    """Overlapping Bronze loads can carry the same (city, reading_time) several times:
    keep the most recently ingested one (so a REVISED forecast value wins)."""
    w = Window.partitionBy("city", "reading_time").orderBy(col("bronze_load_timestamp").desc())
    return df.withColumn("rn", F.row_number().over(w)).filter(col("rn") == 1).drop("rn")

In [0]:
# ==========================================
# MERGE INTO helper (returns rows inserted / rows updated from Delta's own metrics)
#
# Silver: match on (city, reading_time).
#   - not matched -> INSERT
#   - matched AND any value differs -> UPDATE (a revised forecast value)
#   - matched and identical -> untouched, so re-running the same input changes NOTHING
#     (not even load_timestamp): Silver looks exactly the same afterwards.
# ==========================================
def _latest_commit(path):
    h = DeltaTable.forPath(spark, path).history(1).select("version", "operationMetrics").first()
    return h["version"], h["operationMetrics"]

def merge_with_metrics(path, source_df, match_condition, update_condition=None):
    v_before, _ = _latest_commit(path)
    m = DeltaTable.forPath(spark, path).alias("t").merge(source_df.alias("s"), match_condition)
    if update_condition:
        m = m.whenMatchedUpdateAll(condition=update_condition)
    m.whenNotMatchedInsertAll().execute()
    v_after, metrics = _latest_commit(path)
    if v_after == v_before:            # nothing to write -> no new commit
        return 0, 0
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))

SILVER_KEY_COND = "t.city = s.city AND t.reading_time = s.reading_time"
SILVER_CHANGE_COND = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in VALUE_COLS)
QUARANTINE_KEY_COND = ("t.city <=> s.city AND t.reading_time <=> s.reading_time "
                       "AND t.quarantine_reason <=> s.quarantine_reason")

In [0]:
# ==========================================
# Bronze -> Silver for ONE reading-date window (the parameterized unit of work)
# ==========================================
def process_bronze_to_silver(start_date=None, end_date=None, bronze_table_path=None,
                             silver_table_path=None, load_type="incremental"):
    layer = "Bronze-to-Silver"
    bronze_table_path = bronze_table_path or bronze_delta_path
    silver_table_path = silver_table_path or silver_delta_path
    start_time = datetime.now()
    status, error, inserted, updated, n_bad = "Failure", None, 0, 0, 0
    param = "unresolved"
    print(f"--- {layer} [{load_type}] ---")

    try:
        load_ts = datetime.now()
        bronze_all = spark.read.format("delta").load(bronze_table_path)
        new_wm = None
        if start_date and end_date:                       # BACKFILL: by reading date, ignores the watermark
            param = f"{start_date}_to_{end_date}"
            bronze_in = bronze_all
        else:                                             # INCREMENTAL: only Bronze rows not yet processed
            wm = get_watermark()
            bronze_in = bronze_all.filter(col("load_timestamp") > F.lit(wm)) if wm else bronze_all
            new_wm = bronze_in.agg(F.max("load_timestamp")).first()[0]
            param = f"bronze_load_timestamp in ({wm}, {new_wm}]"
            if new_wm is None:
                status = "Success"
                print("No new Bronze data since the last run. Nothing to do.")
                return status
            bronze_in = bronze_in.filter(col("load_timestamp") <= F.lit(new_wm))   # fixed snapshot
        flagged = clean_and_explode(bronze_in, start_date, end_date)

        # 1) bad records -> Silver quarantine table (de-duplicated, idempotent merge)
        bad_df = (flagged.filter(col("dq_reason").isNotNull())
                  .withColumn("quarantine_reason", col("dq_reason"))
                  .withColumn("load_timestamp", F.lit(load_ts))
                  .select(*silver_quarantine_schema.fieldNames())
                  .dropDuplicates(["city", "reading_time", "quarantine_reason"]))
        n_bad = bad_df.count()
        if n_bad > 0:
            merge_with_metrics(silver_quarantine_path, bad_df, QUARANTINE_KEY_COND)

        # 2) valid records -> de-duplicate -> MERGE INTO Silver
        good_df = (dedupe_latest(flagged.filter(col("dq_reason").isNull()))
                   .withColumn("load_timestamp", F.lit(load_ts))
                   .select(*silver_schema.fieldNames()))
        inserted, updated = merge_with_metrics(silver_table_path, good_df,
                                               SILVER_KEY_COND, SILVER_CHANGE_COND)

        if new_wm is not None:
            set_watermark(new_wm)                         # advance only after a successful merge

        if n_bad > 0:
            status = "Partial Success"
            error = (f"{n_bad} rows quarantined in silver_quarantine ({param}); "
                     f"valid rows from the same batch were merged into Silver")
        else:
            status = "Success"
        print(f"{status}: inserted={inserted}, updated={updated}, quarantined={n_bad}")

    except Exception as e:
        error = str(e)[:1000]
        print(f"FAILED: {error}")

    finally:
        write_log(layer, param, load_type, start_time, status,
                  rows_inserted=inserted, rows_updated=updated, rows_quarantined=n_bad,
                  error_message=error)
    return status

In [0]:
# ==========================================
# Parameterized runner: incremental window OR backfill of any period
#
#   mode="incremental" -> everything in Bronze that Silver has not processed yet (watermark);
#                         no dates needed. Re-fetched days arrive as new Bronze loads, so
#                         revised forecast values are picked up and MERGED automatically.
#   mode="backfill"    -> explicit start_date..end_date, processed in `chunk_days` chunks;
#                         each chunk logs its own row, a failed chunk can be re-run alone
# ==========================================
def backfill_silver(start_date, end_date, chunk_days=90):
    current, end = date.fromisoformat(start_date), date.fromisoformat(end_date)
    results = []
    while current <= end:
        chunk_end = min(current + timedelta(days=chunk_days - 1), end)
        status = process_bronze_to_silver(current.isoformat(), chunk_end.isoformat(),
                                          load_type="backfill")
        results.append((current.isoformat(), chunk_end.isoformat(), status))
        current = chunk_end + timedelta(days=1)
    return results

def run_bronze_to_silver(mode="incremental", start_date=None, end_date=None, chunk_days=90):
    if mode == "incremental":
        return [("unprocessed Bronze data", process_bronze_to_silver(load_type="incremental"))]
    if mode == "backfill":
        if not (start_date and end_date):
            raise ValueError("backfill needs start_date and end_date (YYYY-MM-DD)")
        return backfill_silver(start_date, end_date, int(chunk_days))
    raise ValueError("mode must be 'incremental' or 'backfill'")

In [0]:
# ==========================================
# RUN: set the widgets at the top (or pass them as job parameters)
#   Daily load : mode=incremental (leave dates empty)
#   Backfill   : mode=backfill, start_date=2023-10-02, end_date=2026-10-02
# ==========================================
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("chunk_days", "90")

results = run_bronze_to_silver(
    mode=dbutils.widgets.get("mode"),
    start_date=dbutils.widgets.get("start_date") or None,
    end_date=dbutils.widgets.get("end_date") or None,
    chunk_days=dbutils.widgets.get("chunk_days") or 90,
)
for r in results:
    print(r)

In [0]:
# ==========================================
# Evidence: Silver health + audit log
# ==========================================
silver = spark.read.format("delta").load(silver_delta_path)
total = silver.count()
distinct_keys = silver.select("city", "reading_time").distinct().count()
print(f"Silver rows: {total:,} | distinct (city, reading_time): {distinct_keys:,} | duplicates: {total - distinct_keys}")
print("Quarantined rows:", spark.read.format("delta").load(silver_quarantine_path).count())

display(spark.read.format("delta").load(log_table_path).orderBy(col("start_time").desc()))

In [0]:
# ==========================================
# PROOF 1: re-running the same input creates no duplicates and changes nothing.
# Expect run 2 to log rows_inserted=0 and rows_updated=0, and identical row counts.
# ==========================================
silver = spark.read.format("delta").load(silver_delta_path)
latest_day = silver.agg(F.max(F.to_date("reading_time"))).first()[0]
w_start, w_end = str(latest_day - timedelta(days=1)), str(latest_day)

process_bronze_to_silver(w_start, w_end, load_type="rerun_test")
c1 = spark.read.format("delta").load(silver_delta_path).count()
process_bronze_to_silver(w_start, w_end, load_type="rerun_test")      # identical input
c2 = spark.read.format("delta").load(silver_delta_path).count()
print(f"Rows after run 1: {c1:,} | after run 2: {c2:,} | identical: {c1 == c2}")

# ==========================================
# PROOF 2: a REVISED value is updated in place (TA feedback #1), not duplicated.
# We simulate "Silver holds an old forecast value" by overwriting one CO2 value,
# then re-run the same window. Expect rows_updated=1, rows_inserted=0, value restored.
# ==========================================
row = (spark.read.format("delta").load(silver_delta_path)
       .filter(col("reading_time") >= f"{w_start}T00:00:00")
       .orderBy("city", "reading_time").first())
key = f"city = '{row['city']}' AND reading_time = TIMESTAMP '{row['reading_time']}'"
print("Original CO2:", row["carbon_dioxide_ppm"])

DeltaTable.forPath(spark, silver_delta_path).update(condition=key, set={"carbon_dioxide_ppm": "1.0"})
print("Stale value planted:", spark.read.format("delta").load(silver_delta_path).filter(key).first()["carbon_dioxide_ppm"])

process_bronze_to_silver(w_start, w_end, load_type="rerun_test")
print("After MERGE:", spark.read.format("delta").load(silver_delta_path).filter(key).first()["carbon_dioxide_ppm"])

display(spark.read.format("delta").load(log_table_path)
        .filter(col("layer_processed") == "Bronze-to-Silver")
        .orderBy(col("start_time").desc()).limit(3))